In [1]:
import pandas as pd 
import numpy as np
import random
import os 
import argparse
import json
import torch
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.utils.data import TensorDataset
from attrdict import AttrDict
from transformers import BertConfig, BertTokenizer, BertModel
from transformers import DistilBertModel, DistilBertTokenizer, DistilBertConfig, DistilBertForSequenceClassification
from transformers import RobertaModel, RobertaTokenizer, RobertaConfig, RobertaForSequenceClassification
from transformers import AdamW, get_linear_schedule_with_warmup

In [2]:
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="3"

In [3]:
default_path = os.getcwd()
data_path = os.path.join(default_path, '../data')
base_model = os.path.join(default_path, '../base-model')
config_path = os.path.join(default_path, '../config')
log_path = os.path.join(default_path, '../log')
model_path = os.path.join(default_path, '../models')
config_file = "bert-base.json"

In [66]:
criteria = 9; seed = 42; fold = 4

In [67]:
fold_train = pd.read_csv(os.path.join(data_path, f'a{criteria}_score_fold{fold}_train.csv'))
fold_test = pd.read_csv(os.path.join(data_path, f'a{criteria}_score_fold{fold}_test.csv'))

In [68]:
fold_train

,text,text_kor,label
0,I am lucky in that I have not had suicidal tho...,나는 몇 년 동안 자살 생각을 하지 않았다는 점에서 운이 좋다. 나는 Sertral...,1
1,Every part of you just wants to die so badly t...,당신의 모든 부분은 너무 죽고 싶어해서 자신의 생각조차 통제할 수 없다,5
2,I know I am being overdramatic but I have been...,나는 내가 너무 극적이라는 것을 알지만 나는 자주 자살 생각을 하고 있다,8
3,Would it be as painful as I think it would be?...,제가 생각하는 것만큼 고통스러울까요? 나는 항상 스스로 목숨을 끊는 것이 어떤 것인...,4
4,There is a part of me that desperately wants t...,내가 죽고 싶어하는 부분이 있고 나를 여기까지 밀어내기 위해 공모했다,4
...,...,...,...
955,I tried to kill some time by skateboarding but...,"스케이트보드를 타며 시간을 보내려고 했지만, 나는 스케이트보드를 못 타 15분 이상...",0
956,Its ok to want to die of old age?,늙어서 죽고 싶어도 괜찮은가요?,3
957,Have any of you stopped having suicidal though...,여러분 중 자살 생각을 멈추고 기분이 나아진 사람이 있나요? 그렇다면 무엇이 도움이...,2
958,I really do not know that I actually want to die,나는 내가 실제로 죽고 싶어하는지 정말 모르겠다,3


In [69]:
tokenizer = BertTokenizer.from_pretrained(os.path.join(base_model, 'bert-base'), model_max_length=128)
config = BertConfig.from_pretrained(os.path.join(base_model, 'bert-base', 'bert_config.json'))
model = BertModel.from_pretrained(os.path.join(base_model, 'bert-base'), config=config)

In [70]:
with open(os.path.join(config_path, 'training_config.json')) as f:
    training_config = AttrDict(json.load(f))

In [71]:
training_config.seed = seed 

In [72]:
training_config.device = torch.device("cuda") if torch.cuda.is_available() else "cpu"

In [73]:
training_config.config_path = config_path
training_config.log_path = log_path
training_config.data_path = data_path
training_config.model_path = model_path 
training_config.pad = 'max_length'
training_config.num_epochs = 10
training_config.learning_rate = 5e-06
config.max_position_embeddings = 128
config.max_position_embeddings

128

In [74]:
class BertDataset(Dataset):
    def __init__(self, data_file):
        self.data = data_file
    
    def __len__(self):
        return len(self.data.label)
    
    def reset_index(self):
        self.data.reset_index(inplace=True, drop=True)
    
    def __getitem__(self, idx):
        '''
        return text, label
        '''
        self.reset_index()
        text = self.data.text[idx]
        label = self.data.label[idx]
        return text, label

In [75]:
class BertProcessor():
    def __init__(self, config, training_config, tokenizer, truncation=True):
        self.tokenizer = tokenizer 
        self.max_len = config.max_position_embeddings
        self.pad = training_config.pad
        self.batch_size = training_config.train_batch_size
        self.truncation = truncation
    
    def convert_data(self, data_file):
        context2 = None    # single sentence classification
        batch_encoding = self.tokenizer.batch_encode_plus(
            [(data_file[idx][0], context2) for idx in range(len(data_file))],   # text, 
            max_length = self.max_len,
            padding = self.pad,
            truncation = self.truncation
        )
        
        features = []
        for i in range(len(data_file)):
            inputs = {k: batch_encoding[k][i] for k in batch_encoding}
            try:
                inputs['label'] = data_file[i][1] 
            except:
                # print('input label 오류')
                inputs['label'] = 0 
            features.append(inputs)
        
        all_input_ids = torch.tensor([f['input_ids'] for f in features], dtype=torch.long)
        all_attention_mask = torch.tensor([f['attention_mask'] for f in features], dtype=torch.long)
        all_token_type_ids = torch.tensor([f['token_type_ids'] for f in features], dtype=torch.long)
        all_labels = torch.tensor([f['label'] for f in features], dtype=torch.long)

        dataset = TensorDataset(all_input_ids, all_attention_mask, all_token_type_ids, all_labels)
        return dataset
    
    def shuffle_data(self, dataset, data_type):
        if data_type == 'train':
            return RandomSampler(dataset)
        elif data_type == 'eval' or data_type == 'test':
            return SequentialSampler(dataset)
        
    def load_data(self, dataset, sampler):
        return DataLoader(dataset, sampler=sampler, batch_size=self.batch_size)

In [76]:
def RMSELoss(yhat,y):
    return torch.sqrt(torch.mean((yhat-y)**2))

In [77]:
class BertRegressor(nn.Module):
    def __init__(self, config, model):
        super(BertRegressor, self).__init__()
        self.model = model
        self.linear = nn.Linear(config.hidden_size, 128)
        self.relu = nn.ReLU()
        self.out = nn.Linear(128, 1)
    
    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        logits = outputs.last_hidden_state[:, 0, :]
        # print(f'logits: {len(logits)}, {len(logits[0])}')
        x = self.linear(logits)
        x = self.relu(x)
        score = self.out(x)
        # print(f'score: {score}')
        return score 

In [78]:
class BertTrainer():
    def __init__(self, config, training_config, model, train_dataloader, eval_dataloader):
        self.config = config
        self.training_config = training_config
        self.model = model
        self.train_dataloader = train_dataloader
        self.eval_dataloader = eval_dataloader
        
    def set_seed(self):
        random.seed(self.training_config.seed)
        np.random.seed(self.training_config.seed)
        torch.manual_seed(self.training_config.seed)
        if not self.training_config.no_cuda and torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.training_config.seed)
    
    def train(self):
        global_step = 0; nb_eval_steps = 0
        train_rmse = []; eval_rmse = []
        t_total = len(self.train_dataloader) // self.training_config.gradient_accumulation_steps * self.training_config.num_epochs

        optimizer = AdamW(self.model.parameters(), lr=self.training_config.learning_rate, eps=self.training_config.adam_epsilon)
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(t_total * self.training_config.warmup_proportion), \
                                                    num_training_steps=t_total)
        
        criterion = RMSELoss
        # criterion = nn.MSELoss()
        best_loss = 9999 
        
        self.model.zero_grad()
        for epoch in range(int(self.training_config.num_epochs)):
            train_loss = 0.0; eval_loss = 0.0 
            
            for step, batch in enumerate(self.train_dataloader):
                self.model.train()
                batch = tuple(t.to(self.training_config.device) for t in batch)
                inputs = {
                    "input_ids": batch[0],
                    "attention_mask": batch[1],
                    "token_type_ids": batch[2],
                }
                outputs = self.model(**inputs)
                # print(f'output: {type(outputs)}, {outputs.squeeze}')
                label = batch[3]
                # print(f'label: {label}')
                # print(f'output: {outputs}, {outputs.squeeze()}')
                loss = criterion(outputs.squeeze(), batch[3].type_as(outputs))
                loss.backward()
                
                train_loss += loss.item()
                # torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.training_config.max_grad_norm)
                optimizer.step()
                scheduler.step()
                
                self.model.zero_grad()
            
            print(f'epoch: {epoch + 1} done, train_loss: {train_loss / len(self.train_dataloader)}')
            train_rmse.append(train_loss / len(self.train_dataloader))

            for step2, batch2 in enumerate(self.eval_dataloader):
                self.model.eval()
                batch2 = tuple(t.to(self.training_config.device) for t in batch2)

                with torch.no_grad():
                    inputs = {
                        "input_ids": batch2[0],
                        "attention_mask": batch2[1],
                        "token_type_ids": batch2[2],
                    }
                    label2 = batch2[3]
                    outputs = self.model(**inputs)
                    # print(f'eval output: {outputs.squeeze()}, eval label: {label2.type_as(outputs)}')
                    tmp_eval_loss = criterion(outputs.squeeze(), label2.type_as(outputs))
                    eval_loss += tmp_eval_loss.mean().item()
                    
                nb_eval_steps += 1

            eval_loss = eval_loss / nb_eval_steps
            eval_rmse.append(eval_loss)
            if eval_loss < best_loss:
                best_loss = eval_loss
                es = 0
                print(f'save best loss state model & log(epoch {epoch + 1})')
                self.save_model(os.path.join(self.training_config.default_path, self.training_config.model_path, f'bert_bws_{epoch}.pt'))
            else:
                es += 1
                print("Counter {} of 5".format(es))

            if es > 2:
                print("Early stopping with best_loss: ", best_loss, "and val_loss for this epoch: ", eval_loss, "...")
                # self.save_model(os.path.join(self.training_config.default_path, self.training_config.model_path, f'bert_bws_best.pt'))
                break
        # self.save_log(train_rmse, eval_rmse, epoch+1)
        # self.save_model(os.path.join(self.training_config.default_path, self.training_config.model_path, f'bert_bws_a1_10.pt'))
        return train_rmse, eval_rmse

    def save_log(self, train_mse, eval_mse, epoch):
        with open(os.path.join(self.training_config.default_path, self.training_config.log_path, f'train_{epoch}_mse.pickle'), 'wb') as f:
            pickle.dump(train_mse, f, pickle.HIGHEST_PROTOCOL)  
        
        with open(os.path.join(self.training_config.default_path, self.training_config.log_path, f'eval_{epoch}_mse.pickle'), 'wb') as f:
            pickle.dump(eval_mse, f, pickle.HIGHEST_PROTOCOL)  
    
    def save_model(self, model_name):
        torch.save(self.model.state_dict(), model_name)

In [79]:
bert_processor = BertProcessor(config, training_config, tokenizer)

In [80]:
train_file = BertDataset(fold_train)
val_file = BertDataset(fold_test)

In [81]:
train_dataset = bert_processor.convert_data(train_file)
val_dataset = bert_processor.convert_data(val_file)
# test_dataset = bert_processor.convert_data(test_file)

In [82]:
train_sampler = bert_processor.shuffle_data(train_dataset, 'train')
val_sampler = bert_processor.shuffle_data(val_dataset, 'eval')
# test_sampler = bert_processor.shuffle_data(test_dataset, 'test')

In [83]:
train_dataloader = bert_processor.load_data(train_dataset, train_sampler)
val_dataloader = bert_processor.load_data(val_dataset, val_sampler)
# test_dataloader = bert_processor.load_data(test_dataset, test_sampler)

In [84]:
model_reg = BertRegressor(config, model).to(training_config.device)  

In [85]:
bert_model = f'bert_untagged_a{criteria}_s{seed}.pt'

In [86]:
model_reg.load_state_dict(torch.load(os.path.join(model_path, bert_model)))
model_reg.to(training_config.device)

BertRegressor(
  (model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=Tru

In [87]:
bert_trainer = BertTrainer(config, training_config, model_reg, train_dataloader, val_dataloader)

In [88]:
train_mse, eval_mse = bert_trainer.train()

/home/pbl1/anaconda3/envs/aaai/lib/python3.8/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


epoch: 1 done, train_loss: 1.0723988550404708
save best loss state model & log(epoch 1)
epoch: 2 done, train_loss: 0.9661697005232175
save best loss state model & log(epoch 2)
epoch: 3 done, train_loss: 0.9005432209620873
save best loss state model & log(epoch 3)
epoch: 4 done, train_loss: 0.8185667989154657
save best loss state model & log(epoch 4)
epoch: 5 done, train_loss: 0.780098982527852
save best loss state model & log(epoch 5)
epoch: 6 done, train_loss: 0.7424946853270133
save best loss state model & log(epoch 6)
epoch: 7 done, train_loss: 0.7064347736537456
save best loss state model & log(epoch 7)
epoch: 8 done, train_loss: 0.676627521092693
save best loss state model & log(epoch 8)
epoch: 9 done, train_loss: 0.6617357968042294
save best loss state model & log(epoch 9)
epoch: 10 done, train_loss: 0.6516841595371564
save best loss state model & log(epoch 10)


In [89]:
bert_trainer.save_model(os.path.join(model_path, f'bert_untagged_a{criteria}_s{seed}_fold{fold}.pt'))

In [90]:
class BertRegTester():
    def __init__(self, training_config, model):
        self.training_config = training_config
        self.model = model

    def get_label(self, test_dataloader, test_type):
        '''
        test_type: 0  -> Test dataset 
        test_type: 1  -> Test sentence
        '''
        preds = []
        labels = []

        for batch in test_dataloader:
            self.model.eval()    # self 안 붙이면 이상한 Output (BaseModelOutputWithPoolingAndCrossAttentions) 출력 
            batch = tuple(t.to(self.training_config.device) for t in batch)   # args.device: cuda 
            with torch.no_grad():
                inputs = {
                    "input_ids": batch[0],
                    "attention_mask": batch[1],
                    "token_type_ids": batch[2],
                }
                outputs = self.model(**inputs)
                if test_type == 0:
                    preds.extend(outputs.squeeze().detach().cpu().numpy())
                elif test_type == 1:
                    preds.extend(outputs[0].detach().cpu().numpy())            
            label = batch[3].detach().cpu().numpy()
            labels.extend(label)
        return preds, labels 

In [91]:
bert_tester = BertRegTester(training_config, model_reg)

In [92]:
bws_pred, bws_true = bert_tester.get_label(val_dataloader, 0)

In [93]:
bws_pred[:3], bws_true[:3]

([5.500186, 7.2770557, 8.639009], [7, 6, 8])

In [94]:
from torchmetrics import R2Score

criterion = RMSELoss
r2score = R2Score()
bws_rmse = criterion(torch.Tensor(bws_pred), torch.Tensor(bws_true))
bws_r2 = r2score(torch.Tensor(bws_pred), torch.Tensor(bws_true))
bws_rmse, bws_r2

(tensor(1.4203), tensor(0.7147))

In [95]:
bws_pred = pd.DataFrame(bws_pred, columns=['pred'])
bws_pred.head(3)

,pred
0,5.500186
1,7.277056
2,8.639009


In [96]:
bws_pred.to_csv(os.path.join(data_path, f'bert_untagged_a{criteria}_s{seed}_fold{fold}_pred.csv'), index=False)